# Milestone 1: Data Profiling

Profil awal dataset besar Indonesia menggunakan Polars LazyFrame dan DuckDB. Isi sumber serta deskripsi dataset pada `data/README.md` sebelum menjalankan notebook ini.

# 01 - Data Profiling

## Tujuan
Melakukan eksplorasi awal terhadap dataset
Indonesian DFK Domain CPT Corpus.

## Tahapan
- Memuat dataset.
- Memeriksa struktur dan dimensi.
- Mengidentifikasi tipe data.
- Memeriksa missing values dan duplikasi.
- Menganalisis karakteristik panjang teks.
- Melakukan profiling menggunakan Polars Lazy Evaluation.
- Menyimpan dataset dalam format Parquet.

In [ ]:
from datasets import load_from_disk
import polars as pl
from pathlib import Path

DATA_SOURCE = "data/raw/indonesian-dfk-cpt-dataset"
OUTPUT_PATH = Path("data/processed/dataset.parquet")

ds = load_from_disk(DATA_SOURCE)

frames = []

for subset in ds:
    df = pl.from_arrow(ds[subset].data.table)

    df = df.with_columns(
        pl.lit(subset).alias("subset")
    )

    frames.append(df)

dataset = pl.concat(frames, how="vertical")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

dataset.write_parquet(OUTPUT_PATH)

print("Dataset berhasil digabungkan!")
print("Jumlah baris:", f"{dataset.height:,}")
print("Jumlah kolom:", dataset.width)
print("Lokasi:", OUTPUT_PATH)

Dataset berhasil digabungkan!
Jumlah baris: 374,930
Jumlah kolom: 2
Lokasi: data/processed/dataset.parquet


In [ ]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()

DATA_PATH = (
    PROJECT_ROOT /
    "data" /
    "processed" /
    "dataset.parquet"
)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset tidak ditemukan: {DATA_PATH}"
    )

print(f"Dataset: {DATA_PATH.name}")
print(
    f"Ukuran file: "
    f"{DATA_PATH.stat().st_size / 1_000_000:.2f} MB"
)

Dataset: dataset.parquet
Ukuran file: 275.59 MB


## Profil Polars

`scan_parquet` hanya membangun rencana query; file baru dibaca ketika `collect()` dipanggil.

In [ ]:
lazy_data = pl.scan_parquet(DATA_PATH)

print(lazy_data.collect_schema())

ringkasan = lazy_data.select([
    pl.len().alias("jumlah_baris"),
    pl.all().null_count()
]).collect()

ringkasan

Schema({'text': String, 'subset': String})


jumlah_baris,text,subset
u32,u32,u32
374930,0,0


## Profil DuckDB

DuckDB membaca file Parquet langsung dari disk dan menghasilkan ringkasan statistik setiap kolom.

In [ ]:
connection = duckdb.connect()

profil_duckdb = pl.from_pandas(
    connection.execute(
        """
        SUMMARIZE SELECT *
        FROM read_parquet(?)
        """,
        [str(DATA_PATH)]
    ).fetchdf()
)

profil_duckdb

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,f64
"""text""","""VARCHAR""",""" - Bupati Subang Ruhimat punya…","""（) adalah seorang tuan tanah a…",380639,null,null,null,null,null,374930,0.0
"""subset""","""VARCHAR""","""jurnal_dfk""","""wikipedia_general""",6,null,null,null,null,null,374930,0.0


## Catatan Milestone 1

Tulis temuan awal di bawah ini: ukuran data, jumlah baris, tipe kolom, kolom dengan null terbanyak, serta anomali yang perlu ditangani pada Milestone 2.